# Traffic Signal Optimizer - YOLOv8n Fine-tuning (Kaggle)

Train YOLOv8n on the merged 7-class vehicle dataset.

**Before running:**
1. Notebook settings (right panel): **Accelerator = GPU T4 x2** (or P100).
2. **Internet = On** (needed only for the one-time `pip install`).
3. Add the `traffic_yolo` dataset (uploaded to your Kaggle Datasets) via **Add Input**.

**Outputs to download when done:** `yolov8n_traffic_best.pt`, plus `results.png`, `confusion_matrix.png`, `PR_curve.png` from the Output panel.


In [ ]:
!nvidia-smi
!pip install -q ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__)
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import glob, shutil, pathlib, yaml

# Locate the uploaded dataset (any folder containing a data.yaml with 7 classes)
cands = glob.glob('/kaggle/input/**/data.yaml', recursive=True)
assert cands, 'No data.yaml found under /kaggle/input - did you Add Input the traffic_yolo dataset?'
print('candidates:', cands)

src = pathlib.Path(cands[0]).parent
print('using dataset at:', src)

# Kaggle input is read-only; ultralytics writes .cache next to labels, so copy to working
dst = pathlib.Path('/kaggle/working/traffic')
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)

# Rewrite data.yaml with absolute, unambiguous paths
cfg = yaml.safe_load((dst / 'data.yaml').read_text())
cfg['path'] = str(dst)
cfg['train'] = 'train/images'
cfg['val'] = 'val/images'
cfg['test'] = 'test/images'
(dst / 'data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print((dst / 'data.yaml').read_text())

import os
for s in ['train', 'val', 'test']:
    n = len(os.listdir(dst / s / 'images'))
    print(f'{s}: {n} images')

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data='/kaggle/working/traffic/data.yaml',
    epochs=100,
    imgsz=640,
    batch=32,
    patience=20,
    workers=4,
    device=0,
    seed=42,
    pretrained=True,
    cos_lr=True,
    amp=True,
    project='/kaggle/working/runs',
    name='traffic_yolov8n',
    exist_ok=True,
)

In [ ]:
# Evaluate the best checkpoint on the held-out test split
from ultralytics import YOLO

best = '/kaggle/working/runs/traffic_yolov8n/weights/best.pt'
model = YOLO(best)
m = model.val(
    data='/kaggle/working/traffic/data.yaml',
    split='test',
    imgsz=640,
    batch=32,
    plots=True,
    project='/kaggle/working/runs',
    name='eval_final',
)
print(f'mAP@0.5      : {m.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {m.box.map:.4f}')

In [ ]:
import shutil, glob

shutil.copy('/kaggle/working/runs/traffic_yolov8n/weights/best.pt',
            '/kaggle/working/yolov8n_traffic_best.pt')

for pat in ['/kaggle/working/runs/traffic_yolov8n/*.png',
            '/kaggle/working/runs/traffic_yolov8n/weights/*.pt',
            '/kaggle/working/runs/eval_final/*.png']:
    for f in glob.glob(pat):
        try:
            shutil.copy(f, '/kaggle/working/')
        except Exception as e:
            print('skip', f, e)

print('\nFiles in /kaggle/working:')
for f in sorted(glob.glob('/kaggle/working/*')):
    print(' ', f)
print('\nDownload yolov8n_traffic_best.pt + *.png from the Output panel (right side).')